In [1]:
import json
import numpy as np
# file_path = 'results-3/3-chaps-3-books-all-models.json'
# file_path = 'results-3/max_skills=5&data_min=100000&n_repeats=5&n_comps=1000.json'
# file_path = 'results-3/total_irs=106449&max_skills=5&data_min=100000&n_repeats=1&n_comps=1000.json'
file_path = 'results-4/user_count=1000&max_skills=8&n_repeats=5&n_comps=1000&split_type=chronological.json'
with open(file_path, 'r') as f:
    results = json.load(f)

In [2]:
file_path = 'results-5/user_count=1000&max_skills=8&n_repeats=5&n_comps=256&split_type=chronological.json'
with open(file_path, 'r') as f:
    results_mola = json.load(f)

# Use latest results for MoLA, keep other models the same
for k in results_mola:
    results[k]['MoLA'] = results_mola[k]['MoLA']

In [4]:
import numpy as np
import pandas as pd

book_titles = [
    'Chemistry: A Molecular Approach',
    'Microbiology: An Introduction',
    'University Physics with Modern Physics'
]

models = ['MoLA', 'DINA', 'GDINA', 'HO-DINA']
tables = {}

for book_title in book_titles:
    rows = []
    
    for model in models:
        runs = results[book_title][model]
        n = len(runs)
        
        # Helper function for mean and CI string
        def get_stats(data_key):
            values = [r[data_key] for r in runs]
            mean = np.mean(values)
            std_err = np.std(values) / np.sqrt(n)
            ci = 1.96 * std_err
            return mean, ci

        # Calculate Test Stats with CIs
        test_nll_m, test_nll_ci = get_stats('test_nll')
        test_auc_m, test_auc_ci = get_stats('test_auc')
        test_brier_m, test_brier_ci = get_stats('test_brier')

        # Calculate Train Stats (Means only for diagnostic context)
        train_nll = np.mean([r['train_nll'] for r in runs])
        train_auc = np.mean([r['train_auc'] for r in runs])
        train_brier = np.mean([r['train_brier'] for r in runs])

        row = {
            'Model': model,
            'Test AUC (95% CI)': f"{test_auc_m:.3f} (±{test_auc_ci:.3f})",
            'Test NLL (95% CI)': f"{test_nll_m:.3f} (±{test_nll_ci:.3f})",
            'Test Brier (95% CI)': f"{test_brier_m:.3f} (±{test_brier_ci:.3f})",
            'Train AUC': round(train_auc, 3),
            'Train NLL': round(train_nll, 3),
            'Train Brier': round(train_brier, 3)
        }
        rows.append(row)
    
    df = pd.DataFrame(rows).set_index('Model')
    tables[book_title] = df


## Vertical Stack

In [76]:
import pandas as pd
import numpy as np

# 1. Define columns for display
latex_rows = []

for book in book_titles:
    df_book = tables[book].copy()
    
    # --- STEP A: Extract numeric values to find the best models ---
    # We split the string "0.850 (±0.021)" by space and take the first part
    auc_values = df_book['Test AUC (95% CI)'].str.split(' ').str[0].astype(float)
    nll_values = df_book['Test NLL (95% CI)'].str.split(' ').str[0].astype(float)
    brier_values = df_book['Test Brier (95% CI)'].str.split(' ').str[0].astype(float)
    
    # Identify indices for best performance
    best_auc_idx = auc_values.idxmax()
    best_nll_idx = nll_values.idxmin()
    best_brier_idx = brier_values.idxmin()

    # --- STEP B: Build the LaTeX Rows ---
    # Add a spanning header for the Book title
    latex_rows.append(f"\\midrule\n\\multicolumn{{6}}{{c}}{{\\textbf{{{book}}}}} \\\\ \n\\midrule")

    for model_name, row in df_book.iterrows():
        # Get existing formatted strings
        auc_str = row['Test AUC (95% CI)']
        nll_str = row['Test NLL (95% CI)']
        brier_str = row['Test Brier (95% CI)']

        # Apply LaTeX bolding to the best values
        if model_name == best_auc_idx:
            auc_str = f"\\textbf{{{auc_str}}}"
        if model_name == best_nll_idx:
            nll_str = f"\\textbf{{{nll_str}}}"
        if model_name == best_brier_idx:
            brier_str = f"\\textbf{{{brier_str}}}"

        # Clean Train values for consistent decimals (3 places)
        tr_auc = f"{float(row['Train AUC']):.3f}"
        tr_nll = f"{float(row['Train NLL']):.3f}"

        # Final row construction
        row_str = (f"{model_name} & {auc_str} & {nll_str} & {brier_str} & "
                   f"{tr_auc} & {tr_nll} \\\\")
        latex_rows.append(row_str)

# 2. Final Assembly with Table Environment
table_body = "\n".join(latex_rows)

final_latex = f"""
\\begin{{table}}[ht]
\\centering
\\small
\\setlength{{\\tabcolsep}}{{3pt}}
\\caption{{Model Performance Comparison. Best values for each book are in \\textbf{{bold}}.}}
\\label{{tab:results_bolded}}
\\begin{{tabular}}{{lccccc}}
\\toprule
\\textbf{{Model}} & \\textbf{{AUC (95\\% CI)}} & \\textbf{{NLL}} & \\textbf{{Brier}} & \\textbf{{Tr. AUC}} & \\textbf{{Tr. NLL}} \\\\
{table_body}
\\bottomrule
\\end{{tabular}}
\\end{{table}}
"""

print(final_latex)



\begin{table}[ht]
\centering
\small
\setlength{\tabcolsep}{3pt}
\caption{Model Performance Comparison. Best values for each book are in \textbf{bold}.}
\label{tab:results_bolded}
\begin{tabular}{lccccc}
\toprule
\textbf{Model} & \textbf{AUC (95\% CI)} & \textbf{NLL} & \textbf{Brier} & \textbf{Tr. AUC} & \textbf{Tr. NLL} \\
\midrule
\multicolumn{6}{c}{\textbf{Chemistry: A Molecular Approach}} \\ 
\midrule
MoLA & \textbf{0.672 (±0.000)} & \textbf{0.594 (±0.000)} & \textbf{0.201 (±0.000)} & 0.830 & 0.471 \\
DINA & 0.614 (±0.000) & nan (±nan) & 0.257 (±0.000) & 0.797 & 0.522 \\
GDINA & 0.624 (±0.000) & nan (±nan) & 0.249 (±0.000) & 0.827 & 0.481 \\
HO-DINA & 0.623 (±0.000) & nan (±nan) & 0.250 (±0.000) & 0.829 & 0.476 \\
\midrule
\multicolumn{6}{c}{\textbf{Microbiology: An Introduction}} \\ 
\midrule
MoLA & \textbf{0.811 (±0.000)} & \textbf{0.495 (±0.000)} & \textbf{0.154 (±0.000)} & 0.863 & 0.412 \\
DINA & 0.604 (±0.000) & 0.655 (±0.000) & 0.239 (±0.000) & 0.660 & 0.661 \\
GDINA & 0.665 

## Horizontal Stack

In [5]:
short_names = {
    "Chemistry: A Molecular Approach": "Chemistry",
    "Microbiology: An Introduction": "Microbiology",
    "University Physics with Modern Physics": "Physics"
}

# 1. Configuration & Metrics
metrics = ['Test AUC (95% CI)', 'Test NLL (95% CI)', 'Test Brier (95% CI)']
num_books = len(book_titles)

# 2. Build Clean Header Rows
# Row 1: Dataset Names (Centered over 3 columns each)
header_names = ' & ' + ' & '.join([
    f'\\multicolumn{{3}}{{c}}{{\\textbf{{{short_names[b]}}}}}' 
    for b in book_titles
])

# Build dynamic cmidrules based on number of books
cmidrules = ' '.join([f'\\cmidrule(lr){{{3*i+2}-{3*i+4}}}' for i in range(num_books)])

# Row 2: Metric Names (AUC, NLL, Brier)
header_metrics = '\\textbf{Model} & ' + ' & '.join(['AUC' if 'AUC' in m else 'NLL' if 'NLL' in m else 'Brier' for m in metrics] * num_books)

# 3. Build Data Rows with Vertical CIs
data_rows = []
for model in models:
    row_cells = [f"\\textbf{{{model}}}"]
    
    for book in book_titles:
        df_book = tables[book]
        
        # Numeric extraction for bolding
        auc_v = df_book['Test AUC (95% CI)'].str.split(' ').str[0].astype(float)
        nll_v = df_book['Test NLL (95% CI)'].str.split(' ').str[0].astype(float)
        brier_v = df_book['Test Brier (95% CI)'].str.split(' ').str[0].astype(float)
        
        best = {
            'Test AUC (95% CI)': auc_v.idxmax(),
            'Test NLL (95% CI)': nll_v.idxmin(),
            'Test Brier (95% CI)': brier_v.idxmin()
        }
        
        for m in metrics:
            val_str = df_book.loc[model, m] # e.g. "0.850 (±0.021)"
            parts = val_str.split(' ')
            mean_val = parts[0]
            ci_val = parts[1] # e.g. "(±0.021)"
            
            # Stack Mean and CI vertically
            if model == best[m]:
                cell = f"\\shortstack{{\\textbf{{{mean_val}}} \\\\ \\scriptsize {ci_val}}}"
            else:
                cell = f"\\shortstack{{{mean_val} \\\\ \\scriptsize {ci_val}}}"
            row_cells.append(cell)
            
    data_rows.append(" & ".join(row_cells) + " \\\\[0.5ex]") 

# 4. Final Assembly
col_fmt = "l" + "ccc" * num_books

final_latex = f"""
\\begin{{table*}}[ht]
\\centering
\\small
\\setlength{{\\tabcolsep}}{{4pt}} % Increased slightly now that we have more room
\\caption{{Main Results: Performance metrics with stacked 95\\% CIs. Best means per dataset are in \\textbf{{bold}}. Dataset characteristics are summarized in Table~\\ref{{tab:datasets}}.}}
\\label{{tab:stacked_results}}
\\begin{{tabular}}{{{col_fmt}}}
\\toprule
{header_names} \\\\
{cmidrules}
{header_metrics} \\\\
\\midrule
{chr(10).join(data_rows)}
\\bottomrule
\\end{{tabular}}
\\end{{table*}}
"""

print(final_latex)


\begin{table*}[ht]
\centering
\small
\setlength{\tabcolsep}{4pt} % Increased slightly now that we have more room
\caption{Main Results: Performance metrics with stacked 95\% CIs. Best means per dataset are in \textbf{bold}. Dataset characteristics are summarized in Table~\ref{tab:datasets}.}
\label{tab:stacked_results}
\begin{tabular}{lccccccccc}
\toprule
 & \multicolumn{3}{c}{\textbf{Chemistry}} & \multicolumn{3}{c}{\textbf{Microbiology}} & \multicolumn{3}{c}{\textbf{Physics}} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10}
\textbf{Model} & AUC & NLL & Brier & AUC & NLL & Brier & AUC & NLL & Brier \\
\midrule
\textbf{MoLA} & \shortstack{\textbf{0.687} \\ \scriptsize (±0.007)} & \shortstack{\textbf{0.600} \\ \scriptsize (±0.005)} & \shortstack{\textbf{0.206} \\ \scriptsize (±0.002)} & \shortstack{\textbf{0.761} \\ \scriptsize (±0.006)} & \shortstack{\textbf{0.398} \\ \scriptsize (±0.006)} & \shortstack{\textbf{0.123} \\ \scriptsize (±0.002)} & \shortstack{\textbf{0.711} \

## Dataset Stats table

In [103]:
# 1. Dataset Characteristics Table
dataset_rows = []
for b in book_titles:
    n = results[b]["user_count"]
    j = results[b]["item_count"]
    r = results[b]["dataset_size"]
    k = results[b]["skill_count"]
    
    # Calculate density (actual responses / possible responses)
    density = (r / (n * j)) * 100 
    
    row = (
        f"{short_names[b]} & "
        f"{n:,} & "
        f"{j:,} & "
        f"{k} & "
        f"{format_n(r)} & "
        f"{density:.1f}\\% \\\\"
    )
    dataset_rows.append(row)

dataset_latex = f"""
\\begin{{table}}[ht]
\\centering
\\small
\\caption{{Summary of Dataset Characteristics}}
\\label{{tab:datasets}}
\\begin{{tabular}}{{lrrrrr}}
\\toprule
\\textbf{{Dataset}} & \\textbf{{Users ($N$)}} & \\textbf{{Items ($J$)}} & \\textbf{{Skills ($K$)}} & \\textbf{{Resp. ($|R|$)}} & \\textbf{{Density}} \\\\
\\midrule
{chr(10).join(dataset_rows)}
\\bottomrule
\\end{{tabular}}
\\end{{table}}
"""
print(dataset_latex)



\begin{table}[ht]
\centering
\small
\caption{Summary of Dataset Characteristics}
\label{tab:datasets}
\begin{tabular}{lrrrrr}
\toprule
\textbf{Dataset} & \textbf{Users ($N$)} & \textbf{Items ($J$)} & \textbf{Skills ($K$)} & \textbf{Resp. ($|R|$)} & \textbf{Density} \\
\midrule
Chemistry & 1,000 & 14,469 & 243 & 264k & 1.8\% \\
Microbiology & 1,000 & 3,197 & 482 & 217k & 6.8\% \\
Physics & 1,000 & 11,523 & 420 & 222k & 1.9\% \\
\bottomrule
\end{tabular}
\end{table}



In [6]:
tables['University Physics with Modern Physics']

,Test AUC (95% CI),Test NLL (95% CI),Test Brier (95% CI),Train AUC,Train NLL,Train Brier
Model,,,,,,
MoLA,0.711 (±0.004),0.550 (±0.004),0.185 (±0.002),0.799,0.489,0.160
DINA,0.673 (±0.005),0.636 (±0.010),0.209 (±0.002),0.736,0.546,0.186
GDINA,0.702 (±0.003),0.728 (±0.021),0.190 (±0.002),0.771,0.505,0.797
HO-DINA,0.702 (±0.003),0.729 (±0.021),0.190 (±0.002),0.771,0.505,0.797


In [7]:
tables['Microbiology: An Introduction']

,Test AUC (95% CI),Test NLL (95% CI),Test Brier (95% CI),Train AUC,Train NLL,Train Brier
Model,,,,,,
MoLA,0.761 (±0.006),0.398 (±0.006),0.123 (±0.002),0.818,0.355,0.105
DINA,0.628 (±0.004),0.570 (±0.012),0.186 (±0.004),0.665,0.528,0.176
GDINA,0.686 (±0.004),0.506 (±0.003),0.132 (±0.003),0.731,0.427,0.549
HO-DINA,0.687 (±0.004),0.506 (±0.003),0.132 (±0.003),0.731,0.427,0.549


In [8]:
tables['Chemistry: A Molecular Approach']

,Test AUC (95% CI),Test NLL (95% CI),Test Brier (95% CI),Train AUC,Train NLL,Train Brier
Model,,,,,,
MoLA,0.687 (±0.007),0.600 (±0.005),0.206 (±0.002),0.787,0.534,0.178
DINA,0.656 (±0.008),0.748 (±0.018),0.237 (±0.003),0.757,0.537,0.190
GDINA,0.674 (±0.008),1.075 (±0.074),0.224 (±0.003),0.783,0.503,0.871
HO-DINA,0.674 (±0.008),1.076 (±0.073),0.224 (±0.003),0.783,0.503,0.871


## LaTeX Results Table

In [40]:
import numpy as np

book_titles = [
    'Chemistry: A Molecular Approach',
    'Microbiology: An Introduction',
    'University Physics with Modern Physics'
]

short_names = {
    'Chemistry: A Molecular Approach': 'Chemistry',
    'Microbiology: An Introduction': 'Microbiology',
    'University Physics with Modern Physics': 'Physics'
}

metrics = ['Test AUC', 'Test NLL', 'Test Brier']

def format_n(n):
    if n >= 1_000_000:
        return f"{n/1_000_000:.1f}M"
    elif n >= 1_000:
        return f"{int(n/1000)}k"
    return str(n)

latex = []
latex.append(r'\begin{table*}[t]')
latex.append(r'\centering')
latex.append(r'\small')
latex.append(r'\begin{tabular}{lccc|ccc|ccc}')
latex.append(r'\toprule')

# ---- Header row: book names ----
header1 = ' & ' + ' & '.join(
    [f'\\multicolumn{{3}}{{c}}{{{short_names[b]}}}' for b in book_titles]
)
latex.append(header1 + r' \\')

# ---- Header row: dataset info ----
header2 = ' & ' + ' & '.join([
    r'\multicolumn{3}{c}{(' +
    f'N={format_n(results[b]["dataset_size"])}, '
    f'J={format_n(results[b]["item_count"])}, '
    f'K={results[b]["skill_count"]}' +
    r')}'
    for b in book_titles
])
latex.append(header2 + r' \\')

# ---- Header row: metrics ----
header3 = 'Model '
for _ in book_titles:
    header3 += '& AUC & NLL & Brier '
latex.append(header3 + r'\\')

latex.append(r'\midrule')

models = tables[book_titles[0]].index

for model in models:
    row = [model]
    
    for book in book_titles:
        df = tables[book]
        
        best_auc = df['Test AUC'].max()
        best_nll = df['Test NLL'].min()
        best_brier = df['Test Brier'].min()
        
        auc = df.loc[model, 'Test AUC']
        nll = df.loc[model, 'Test NLL']
        brier = df.loc[model, 'Test Brier']
        
        def fmt(val, best, higher_is_better):
            if (higher_is_better and val == best) or (not higher_is_better and val == best):
                return r'\textbf{' + f'{val:.3f}' + '}'
            return f'{val:.3f}'
        
        row.append(fmt(auc, best_auc, True))
        row.append(fmt(nll, best_nll, False))
        row.append(fmt(brier, best_brier, False))
    
    latex.append(' & '.join(row) + r'\\')

latex.append(r'\bottomrule')
latex.append(r'\end{tabular}')
latex.append(r'\caption{Test performance across textbooks. '
             r'Higher AUC is better; lower NLL and Brier are better.}')
latex.append(r'\label{tab:main-results}')
latex.append(r'\end{table*}')

print('\n'.join(latex))

\begin{table*}[t]
\centering
\small
\begin{tabular}{lccc|ccc|ccc}
\toprule
 & \multicolumn{3}{c}{Chemistry} & \multicolumn{3}{c}{Microbiology} & \multicolumn{3}{c}{Physics} \\
 & \multicolumn{3}{c}{(N=29k, J=1k, K=24)} & \multicolumn{3}{c}{(N=19k, J=373, K=84)} & \multicolumn{3}{c}{(N=27k, J=1k, K=49)} \\
Model & AUC & NLL & Brier & AUC & NLL & Brier & AUC & NLL & Brier \\
\midrule
MoLA & \textbf{0.682} & \textbf{0.575} & 0.192 & \textbf{0.668} & 0.439 & \textbf{0.137} & 0.687 & 0.641 & 0.218\\
DINA & 0.669 & 0.621 & 0.202 & 0.664 & \textbf{0.418} & 0.139 & 0.709 & \textbf{0.598} & 0.196\\
GDINA & 0.681 & 0.707 & 0.191 & 0.665 & 0.432 & 0.138 & 0.712 & 0.670 & \textbf{0.193}\\
HO-DINA & 0.681 & 0.707 & \textbf{0.191} & 0.665 & 0.432 & 0.138 & \textbf{0.712} & 0.671 & 0.193\\
\bottomrule
\end{tabular}
\caption{Test performance across textbooks. Higher AUC is better; lower NLL and Brier are better.}
\label{tab:main-results}
\end{table*}
